해시체인 기본코드 추가

In [7]:

import hashlib, json
from datetime import datetime, timezone

GENESIS_HASH = "0" * 64

def canonical(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False)

def compute_hash(seq, prev_hash, record, created_at):
    body = canonical({"seq": seq, "prev_hash": prev_hash,
                      "record": record, "created_at": created_at})
    return hashlib.sha256(body.encode("utf-8")).hexdigest()

def build_entry(prev_entry, record):
    seq = 1 if prev_entry is None else prev_entry["seq"] + 1
    prev_hash = GENESIS_HASH if prev_entry is None else prev_entry["hash"]
    created_at = datetime.now(timezone.utc).isoformat()
    return {"seq": seq, "prev_hash": prev_hash, "record": record,
            "created_at": created_at,
            "hash": compute_hash(seq, prev_hash, record, created_at)}

def verify_chain(entries):
    prev_hash = GENESIS_HASH
    for i, e in enumerate(entries, start=1):
        if e["seq"] != i:
            return {"ok": False, "broken_seq": i, "reason": "순번이 끊김 (기록 삭제·삽입 의심)"}
        if e["prev_hash"] != prev_hash:
            return {"ok": False, "broken_seq": i, "reason": "앞 기록의 지문이 안 맞음"}
        if compute_hash(e["seq"], e["prev_hash"], e["record"], e["created_at"]) != e["hash"]:
            return {"ok": False, "broken_seq": i, "reason": "기록 내용이 바뀜 (변조)"}
        prev_hash = e["hash"]
    return {"ok": True, "count": len(entries)}

In [6]:
import copy
from hashchain import build_entry, verify_chain

chain, prev = [], None
for i in range(1, 4):
    record = {"anon_id": "U7F3A", "job": "hr", "scenario_id": f"S0{i}",
              "choice": "click", "is_correct": False,
              "analysis": {"type": "url", "score": 60}}
    prev = build_entry(prev, record)
    chain.append(prev)

print("1) 정상:", verify_chain(chain))
t = copy.deepcopy(chain); t[1]["record"]["is_correct"] = True
print("2) 2번 기록 수정:", verify_chain(t))
t = copy.deepcopy(chain); del t[1]
print("3) 2번 기록 삭제:", verify_chain(t))

1) 정상: {'ok': True, 'count': 3}
2) 2번 기록 수정: {'ok': False, 'broken_seq': 2, 'reason': '기록 내용이 바뀜 (변조)'}
3) 2번 기록 삭제: {'ok': False, 'broken_seq': 2, 'reason': '순번이 끊김 (기록 삭제·삽입 의심)'}
